<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/18_01_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [32]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2026-01-18 22:41:24--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt.6’

names.txt.6         100%[===================>] 222.80K  --.-KB/s    in 0.03s   

2026-01-18 22:41:24 (7.45 MB/s) - ‘names.txt.6’ saved [228145/228145]



In [33]:
words = open("names.txt", "r").read().splitlines()

In [34]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set('.'.join(words))))
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)


In [35]:
def data_prep(data,block_size):
  data= ["."*block_size +word +"." for word in data]
  Y=[stoi[i] for ch in data for i in ch[block_size:]]
  X=[word[i:i+block_size] for word in data for i in range(len(word)-block_size)]
  X=[stoi[i] for x in X for i in x]
  X=torch.tensor(X).view(-1,block_size)
  Y=torch.tensor(Y)
  return X, Y
block_size=3
Xtr,Ytr=data_prep(words,block_size)


In [36]:
n_embd = 3 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP

In [37]:
class Linear:

  def __init__(self, fan_in, fan_out, bias = True ):
    self.weights=torch.rand(fan_in,fan_out)*fan_in**0.5
    self.weights.requires_grad = True
    if bias == True:
       self.bias = torch.zeros(fan_out)
       self.bias.requires_grad = True
    else:
       self.bias =None


  def __call__(self, x):
    self.out=x@self.weights
    if self.bias is not None:
      self.out+=self.bias
    return(self.out)
  def parameters(self):
     return [self.weights]+ ([] if self.bias is None else [self.bias])


In [38]:
z=Linear(n_embd * block_size, n_hidden, bias=False)
z.weights.shape



torch.Size([9, 100])

In [39]:
C = torch.randn(vocab_size, n_embd)


In [40]:
ix = torch.randint(0, Xtr.shape[0], (32,))
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
C.requires_grad=True
emb = C[Xb] # embed the characters into vectors
x = emb.view(emb.shape[0], -1) # concatenate the vectors
x = z(x)
loss = F.cross_entropy(x, Yb) # loss function

In [41]:
parameters = [C] + [p for p in z.parameters()]


In [47]:
for p in parameters:
    p.grad = None
loss.backward()
for p in parameters:
    p.data +=  p.grad

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [111]:
ix = torch.randint(0, Xtr.shape[0], (32,))
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
C.requires_grad=True
emb = C[Xb] # embed the characters into vectors
x = emb.view(emb.shape[0], -1) # concatenate the vectors
x = z(x)
loss = F.cross_entropy(x, Yb) # loss function
for p in parameters:
    p.grad = None
loss.backward()
for p in parameters:
    p.data -=  0.1*p.grad
loss

tensor(4.1724, grad_fn=<NllLossBackward0>)

tensor([[ 1.7585e+00,  7.6370e-02,  1.3895e+00,  2.1629e+00,  1.1393e+00,
          1.7439e+00,  1.8236e-01,  2.2948e+00,  2.6267e+00,  5.2052e-01,
          1.3601e+00,  1.8284e+00,  6.0108e-01,  1.5763e+00,  7.0773e-01,
          1.8439e+00,  2.2033e+00,  2.2283e+00,  1.1489e+00,  1.8634e+00,
          1.6738e+00,  2.7997e+00,  1.0988e+00,  6.5065e-01,  2.5447e+00,
          4.5264e-01,  2.3735e-01,  1.4274e+00,  1.8497e-01,  1.7141e-01,
          3.6064e-01,  5.3935e-01,  1.1977e+00,  1.9661e+00,  1.5123e+00,
          9.7551e-01,  1.3350e+00,  1.3367e+00,  2.0027e+00,  6.2913e-01,
          1.5108e-01,  7.9180e-01,  1.5061e+00,  1.5314e+00,  2.8259e+00,
          2.2545e+00,  1.9270e+00,  2.0981e-01,  5.8566e-01,  2.9947e+00,
          6.6098e-02,  2.4863e+00,  2.7586e+00,  2.1974e+00,  1.1113e+00,
          1.8980e+00,  2.5034e+00,  3.8035e-01,  1.5266e+00,  2.5827e+00,
          6.2173e-01,  1.0970e+00,  2.0028e+00,  2.2736e+00,  7.4105e-01,
          1.3949e+00,  1.5827e+00,  1.